# Nemotron Reasoning Challenge

End-to-end pipeline for the [NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge). Works on **Kaggle T4 / T4x2** and **Udacity GPU workspaces**.

| Phase | Script | What it does |
|-------|--------|--------------|
| 1 | `scripts/01_eda.py` | Classify prompts, token-length stats, EDA report |
| 2 | `scripts/02_prepare_data.py` | Synthetic data + optional API CoT → `train_sft.jsonl` |
| 3 | `scripts/03_train_lora.py` | QLoRA SFT (4-bit, rank 32) on Nemotron-3-Nano-30B |
| 4 | `scripts/04_evaluate.py` | vLLM inference + accuracy (optional) |
| 5 | `scripts/05_package_submission.py` | Zip LoRA adapter → `submission.zip` |

**Kaggle:** Add the competition, the Nemotron-3 Model input, and a Dataset with `scripts/`. The notebook auto-detects all paths under `/kaggle/input`.

**Udacity / local:** Clone/upload the repo, place `train.csv` in `data/`, and set a HuggingFace token for model download.

In [ ]:
from pathlib import Path
import os

# ---------------------------------------------------------------------------
# Environment detection
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

WORK_ROOT = Path("/kaggle/working/project" if IS_KAGGLE else ".").resolve()

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # or paste token string here

# ---------------------------------------------------------------------------
# GPU profile — T4 (16 GB) defaults
# "t4"    — single T4, 4-bit QLoRA + 2048 ctx
# "t4_x2" — Kaggle T4x2, same training but both GPUs + more CPU staging
# "high_vram" — A100 / large GPU, bf16 --no-quant + longer ctx
# ---------------------------------------------------------------------------
GPU_PROFILE = "t4_x2"

if GPU_PROFILE == "high_vram":
    USE_BF16_FULL = True
    TRAIN_MAX_SEQ = 4096
    TRAIN_MAX_MEMORY_JSON = None
elif GPU_PROFILE == "t4_x2":
    USE_BF16_FULL = False
    TRAIN_MAX_SEQ = 2048
    TRAIN_MAX_MEMORY_JSON = '{"0":"10GiB","cpu":"2GiB","disk":"150GiB"}'
else:
    USE_BF16_FULL = False
    TRAIN_MAX_SEQ = 2048
    TRAIN_MAX_MEMORY_JSON = '{"0":"10GiB","cpu":"2GiB","disk":"150GiB"}'

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
LORA_TARGET_MODE = "kaggle_nemotron"
LORA_ALPHA = 16
TRAIN_BATCH = 1
GRAD_ACCUM = 16
NUM_EPOCHS = 2.0

# ---------------------------------------------------------------------------
# Data preparation
# ---------------------------------------------------------------------------
SKIP_COT = True
SYNTHETIC_PER_KIND = 400

# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
RUN_VLLM_EVAL = False

print("IS_KAGGLE:", IS_KAGGLE)
print("WORK_ROOT:", WORK_ROOT)
print("GPU_PROFILE:", GPU_PROFILE, "| USE_BF16_FULL:", USE_BF16_FULL,
      "| TRAIN_MAX_SEQ:", TRAIN_MAX_SEQ)
print("TRAIN_MAX_MEMORY_JSON:", TRAIN_MAX_MEMORY_JSON or "(auto)")

## Install dependencies

On **Kaggle**, downgrades torch to 2.4.0+cu121 so that pre-built `causal-conv1d` / `mamba-ssm` wheels are available (no 1-hour source compile). On **Udacity / local**, keeps the pre-installed torch and downloads matching GitHub release wheels. Key constraint: **transformers < 5** — v5 can OOM on T4 during 4-bit weight materialisation.

In [ ]:
import subprocess, sys, os, re
from pathlib import Path

def pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def pip_try(*args: str) -> bool:
    try:
        pip_install(*args)
        return True
    except subprocess.CalledProcessError:
        return False

pip_install("-U", "pip", "setuptools", "wheel")

# ---------------------------------------------------------------------------
# Kaggle: downgrade torch to 2.4.0+cu121 so pre-built causal-conv1d / mamba-ssm
# wheels are available instantly (no 1-hour source compile).
# Udacity / local: keep pre-installed torch.
# ---------------------------------------------------------------------------
if IS_KAGGLE:
    print("Kaggle: installing torch 2.4.0+cu121 (has published mamba/causal-conv1d wheels)...")
    pip_install(
        "torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0",
        "--index-url", "https://download.pytorch.org/whl/cu121",
    )

import torch
_torch_full = torch.__version__
print(f"torch {_torch_full}")

pip_install(
    "transformers>=4.45,<5",
    "peft>=0.12",
    "trl>=0.12",
    "datasets",
    "accelerate",
    "bitsandbytes",
    "psutil",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
    "huggingface_hub",
    "ninja",
)

# ---------------------------------------------------------------------------
# causal-conv1d + mamba-ssm
# 1) Kaggle (torch 2.4+cu121): pip has prebuilt wheels on PyPI → fast
# 2) Elsewhere: try GitHub release wheels matched to detected torch version
# 3) Last resort: build from source
# ---------------------------------------------------------------------------
_torch_major_minor = re.match(r"(\d+\.\d+)", _torch_full).group(1)
_torch_minor = int(_torch_major_minor.split(".")[1])

if IS_KAGGLE:
    _GH_C = "https://github.com/Dao-AILab/causal-conv1d/releases/download"
    _GH_M = "https://github.com/state-spaces/mamba/releases/download"
    _kpy = f"cp{sys.version_info.major}{sys.version_info.minor}"
    pip_install(f"{_GH_C}/v1.5.4/causal_conv1d-1.5.4+cu12torch2.4cxx11abiFALSE-{_kpy}-{_kpy}-linux_x86_64.whl")
    pip_install(f"{_GH_M}/v2.2.4/mamba_ssm-2.2.4+cu12torch2.4cxx11abiFALSE-{_kpy}-{_kpy}-linux_x86_64.whl")
    print("OK: causal-conv1d + mamba-ssm (GitHub wheels for torch 2.4+cu12)")
else:
    _cu = "cu12" if "cu12" in _torch_full else ("cu11" if "cu11" in _torch_full else "cu12")
    _py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    _abi_order = (["cxx11abiTRUE", "cxx11abiFALSE"] if _torch_minor >= 7
                  else ["cxx11abiFALSE", "cxx11abiTRUE"])
    print(f"Wheel tags: torch{_torch_major_minor}, {_cu}, abi={_abi_order[0]}, {_py}")

    _GH_CAUSAL = "https://github.com/Dao-AILab/causal-conv1d/releases/download"
    _GH_MAMBA  = "https://github.com/state-spaces/mamba/releases/download"

    _pkg_versions = [
        ("causal-conv1d", [
            (_GH_CAUSAL, "v1.6.1", "causal_conv1d", "1.6.1"),
            (_GH_CAUSAL, "v1.5.4", "causal_conv1d", "1.5.4"),
        ]),
        ("mamba-ssm", [
            (_GH_MAMBA, "v2.3.1", "mamba_ssm", "2.3.1"),
            (_GH_MAMBA, "v2.2.4", "mamba_ssm", "2.2.4"),
        ]),
    ]

    for pkg, versions in _pkg_versions:
        installed = False
        for gh_base, tag, whl_name, whl_ver in versions:
            if installed:
                break
            for abi in _abi_order:
                url = (f"{gh_base}/{tag}/{whl_name}-{whl_ver}+{_cu}torch{_torch_major_minor}"
                       f"{abi}-{_py}-{_py}-linux_x86_64.whl")
                if pip_try(url):
                    print(f"OK: {pkg} {whl_ver} ({abi})")
                    installed = True
                    break
        if not installed:
            print(f"No prebuilt wheel for {pkg}; building from source (slow)...")
            os.environ["CAUSAL_CONV1D_FORCE_BUILD"] = "TRUE"
            os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
            pip_install("--no-build-isolation", "--no-deps", pkg)
            print(f"OK: {pkg} (built from source)")

pip_try("--prefer-binary", "nvidia-cutlass>=3.6", "nvidia-cutlass-dsl>=4.4")

if not pip_try("polars"):
    print("polars install skipped (optional).")

if RUN_VLLM_EVAL:
    pip_install("vllm>=0.6")

print("\n--- Installed versions ---")
for pkg in ("torch", "transformers", "peft", "trl", "bitsandbytes", "mamba_ssm"):
    try:
        v = __import__(pkg).__version__
        print(f"  {pkg}: {v}")
    except Exception:
        print(f"  {pkg}: not found")

## Setup workspace and data

**Kaggle:** Scripts and `train.csv` are auto-detected from `/kaggle/input` (competition + dataset inputs).

**Udacity / local:** Place `train.csv` in `WORK_ROOT/data/`, or uncomment the Kaggle API lines below.

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK_ROOT.mkdir(parents=True, exist_ok=True)
data_dir = WORK_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "reports").mkdir(parents=True, exist_ok=True)
(WORK_ROOT / "data" / "synthetic").mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Kaggle: auto-find scripts/ dataset and competition train.csv
# ---------------------------------------------------------------------------
if IS_KAGGLE:
    _kaggle_root = Path("/kaggle/input")

    # Find the dataset that contains scripts/01_eda.py
    _code_source = None
    for d in sorted(_kaggle_root.iterdir()):
        if d.is_dir() and (d / "scripts" / "01_eda.py").is_file():
            _code_source = d
            break
    if _code_source is None:
        try:
            for eda in _kaggle_root.rglob("01_eda.py"):
                if eda.parent.name == "scripts":
                    _code_source = eda.parent.parent.resolve()
                    break
        except OSError:
            pass

    if _code_source is not None:
        for item in ("scripts", "requirements.txt", "requirements-vllm.txt"):
            src = _code_source / item
            if not src.exists():
                continue
            dst = WORK_ROOT / item
            if dst.exists():
                shutil.rmtree(dst) if dst.is_dir() else dst.unlink()
            if src.is_dir():
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
        print("Copied code from", _code_source)
    else:
        print("WARN: No scripts/ dataset found under /kaggle/input. "
              "Add a Dataset containing this repo's scripts/ folder.")

    # Find competition train.csv
    _comp_dirs = [
        _kaggle_root / "competitions" / "nvidia-nemotron-model-reasoning-challenge",
        _kaggle_root / "nvidia-nemotron-model-reasoning-challenge",
        _kaggle_root / "nvidia-nemotron-3-reasoning-challenge",
    ]
    _comp_data = None
    for cd in _comp_dirs:
        if cd.is_dir() and (cd / "train.csv").is_file():
            _comp_data = cd
            break
    if _comp_data is None:
        for d in sorted(_kaggle_root.iterdir()):
            if d.is_dir() and (d / "train.csv").is_file():
                _comp_data = d
                break
    if _comp_data is not None:
        for fname in ("train.csv", "test.csv"):
            src = _comp_data / fname
            if src.is_file():
                shutil.copy2(src, data_dir / fname)
                print("Copied", fname, "from", _comp_data.name)
    else:
        print("WARN: No train.csv found under /kaggle/input. "
              "Add the competition data input.")

# ---------------------------------------------------------------------------
# Non-Kaggle: manual upload or Kaggle API
# ---------------------------------------------------------------------------
if not IS_KAGGLE and not (data_dir / "train.csv").is_file():
    print("train.csv not found. Uncomment the Kaggle API lines below, or "
          "upload train.csv to", data_dir)
    # import subprocess, sys, zipfile
    # subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    # subprocess.check_call([
    #     "kaggle", "competitions", "download",
    #     "-c", "nvidia-nemotron-model-reasoning-challenge",
    #     "-p", str(data_dir),
    # ])
    # for zf in data_dir.glob("*.zip"):
    #     zipfile.ZipFile(zf).extractall(data_dir)
    #     zf.unlink()

os.chdir(WORK_ROOT)
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

assert (WORK_ROOT / "scripts" / "01_eda.py").is_file(), (
    f"scripts/01_eda.py not found under {WORK_ROOT}. "
    "On Kaggle, add a Dataset with scripts/. Otherwise clone the repo."
)
assert (data_dir / "train.csv").is_file(), (
    f"train.csv not found in {data_dir}. "
    "On Kaggle, add the competition data input. "
    "Otherwise download it or uncomment the Kaggle API lines above."
)

print("cwd:", os.getcwd())
print("train.csv:", (data_dir / "train.csv").stat().st_size, "bytes")

## Resolve base model

**Kaggle:** Uses the competition Model input mounted under `/kaggle/input` (no download needed).

**Udacity / local:** Downloads from HuggingFace Hub (~60 GB bf16 safetensors). Requires `HF_TOKEN` with access to the gated model. If disk is tight, set `HF_HOME` to a larger volume.

In [ ]:
import importlib.util
from pathlib import Path

MODEL_PATH_LOCAL = None

# On Kaggle, try the competition Model input first (no download needed)
if IS_KAGGLE:
    _knp_file = WORK_ROOT / "scripts" / "kaggle_nemotron_paths.py"
    if _knp_file.is_file():
        _spec = importlib.util.spec_from_file_location("kaggle_nemotron_paths", _knp_file)
        _knp = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_knp)
        _found = _knp.find_kaggle_competition_nemotron_dir()
        if _found:
            MODEL_PATH_LOCAL = _found
            print("Using Kaggle Model mount:", MODEL_PATH_LOCAL)

# Also check NEMOTRON_MODEL_PATH env var
if MODEL_PATH_LOCAL is None:
    _env = os.environ.get("NEMOTRON_MODEL_PATH", "").strip()
    if _env and Path(_env).is_dir() and (Path(_env) / "config.json").is_file():
        MODEL_PATH_LOCAL = _env
        print("Using NEMOTRON_MODEL_PATH:", MODEL_PATH_LOCAL)

# Fall back to HuggingFace download
if MODEL_PATH_LOCAL is None:
    from huggingface_hub import login, snapshot_download

    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF login: OK")
    else:
        try:
            login(add_to_git_credential=False)
            print("HF login: OK (interactive)")
        except Exception as e:
            print("HF login skipped:", e)

    MODEL_PATH_LOCAL = snapshot_download(MODEL_ID, resume_download=True)
    print("Model cached at:", MODEL_PATH_LOCAL)

print("MODEL_PATH_LOCAL:", MODEL_PATH_LOCAL)

## Phase 1 — EDA

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/01_eda.py",
        "--data-dir", "data",
        "--report-dir", "data/reports",
        "--tokenizer-model", str(MODEL_PATH_LOCAL),
    ],
    check=True,
)

## Phase 2 — Prepare SFT data

In [ ]:
import subprocess, sys

skip = ["--skip-cot"] if SKIP_COT else []
cmd = [
    sys.executable,
    "scripts/02_prepare_data.py",
    "--data-dir", "data",
    "--synthetic-dir", "data/synthetic",
    "--output", "data/train_sft.jsonl",
    "--tokenizer-model", str(MODEL_PATH_LOCAL),
    "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
] + skip
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Phase 3 — LoRA training

QLoRA (4-bit NF4) on T4. `max_memory` caps GPU at 10 GiB and CPU at 2 GiB to avoid OOM during shard loading. On Kaggle, Triton/RMSNorm patches are enabled automatically.

In [ ]:
import gc, os, subprocess, sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

train_env = os.environ.copy()
for k, v in (
    ("OMP_NUM_THREADS", "1"),
    ("MKL_NUM_THREADS", "1"),
    ("OPENBLAS_NUM_THREADS", "1"),
    ("NUMEXPR_NUM_THREADS", "1"),
    ("OMP_WAIT_POLICY", "passive"),
    ("MALLOC_ARENA_MAX", "1"),
    ("TOKENIZERS_PARALLELISM", "false"),
):
    train_env.setdefault(k, v)

if IS_KAGGLE:
    train_env.setdefault("NEMOTRON_KAGGLE_PATCHES", "1")
    if GPU_PROFILE == "t4":
        train_env.setdefault("NEMOTRON_SHARD_LOAD_MAX_CPU_GIB", "3")
    elif GPU_PROFILE == "t4_x2":
        train_env.setdefault("NEMOTRON_SHARD_LOAD_MAX_CPU_GIB", "8")
        train_env.setdefault("NEMOTRON_SKIP_CPU_MAX_MEMORY_CLAMP", "1")
else:
    train_env.setdefault("NEMOTRON_KAGGLE_PATCHES", "0")

offload = str((WORK_ROOT / "lora_output" / "hf_offload").resolve())

cmd = [
    sys.executable,
    "scripts/03_train_lora.py",
    "--data-path", "data/train_sft.jsonl",
    "--output-dir", "lora_adapter",
    "--checkpoint-dir", "lora_output",
    "--offload-folder", offload,
    "--model-path", str(MODEL_PATH_LOCAL),
    "--lora-target-mode", LORA_TARGET_MODE,
    "--lora-alpha", str(LORA_ALPHA),
    "--batch-size", str(TRAIN_BATCH),
    "--grad-accum", str(GRAD_ACCUM),
    "--epochs", str(NUM_EPOCHS),
    "--max-seq-length", str(TRAIN_MAX_SEQ),
    "--force-peft",
    "--dataloader-workers", "0",
]
if not IS_KAGGLE:
    cmd.append("--no-nemotron-kaggle-patches")
if USE_BF16_FULL:
    cmd.append("--no-quant")
if TRAIN_MAX_MEMORY_JSON:
    cmd.extend(["--max-memory-json", TRAIN_MAX_MEMORY_JSON])

gc.collect()
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=train_env)

## Phase 4 — Evaluation (optional)

Requires `vllm`. Set `RUN_VLLM_EVAL = True` in the config cell to enable.

In [ ]:
import subprocess, sys

if RUN_VLLM_EVAL:
    subprocess.run(
        [
            sys.executable,
            "scripts/04_evaluate.py",
            "--adapter-path", "lora_adapter",
            "--data-dir", "data",
            "--max-samples", "32",
        ],
        check=True,
    )
else:
    print("Skipping vLLM eval (set RUN_VLLM_EVAL = True to enable).")

## Phase 5 — Package submission

In [ ]:
import subprocess, sys

subprocess.run(
    [
        sys.executable,
        "scripts/05_package_submission.py",
        "--adapter-dir", "lora_adapter",
        "--output", "submission.zip",
    ],
    check=True,
)

zp = WORK_ROOT / "submission.zip"
print("submission.zip:", zp.is_file(), zp.stat().st_size if zp.is_file() else 0, "bytes")

if IS_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink("submission.zip"))